# 3. Install one VFL method

Run this notebook every time you want to switch method. It:

1. Removes whatever DYNAMOS method is currently installed (Helm releases, namespaces, PVs) and wipes the
   DYNAMOS host data on the `dynamos` VM, so no state leaks between methods.
2. Clones the method's branch onto `control` as `~/sd-<branch>` (or reuses / updates an existing checkout).
3. Runs that branch's own installer (`configuration/dynamos-configuration.sh fabric`), which deploys the images
   `dynamos1/*:<branch>` (or `IMAGE_TAG` if set). The monitoring stack is skipped unless `INSTALL_MONITORING`.
4. Waits until everything is ready and optionally runs a short smoke test.

Requires **02_setup_cluster** to have been completed once.

In [ ]:
METHOD = "baseline"          # "baseline" | "fed-bcd" | "semi-overlap-fed-bcd" | "fed-encrypt-dynamic"
BRANCH = METHOD              # git branch to deploy (change only to test a different branch of the same method)
CLONE_IF_MISSING = True      # clone the branch onto control if ~/sd-<branch> does not exist yet
UPDATE_CHECKOUT = False      # fetch + hard-reset an existing checkout to origin/<branch> (discards remote local changes)
RESET_FIRST = True           # uninstall the current method and wipe DYNAMOS data first (strongly recommended)
IMAGE_TAG = None             # None = the branch default from dynamos.conf (the branch name)
INSTALL_MONITORING = False   # Prometheus/Kepler stack; not needed for the VFL experiments
RUN_SMOKE_TEST = True

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

import deployment_config as cfg
import fabric_helpers as fh
from fabric_helpers import run, run_script

fablib = fablib_manager()
slice_ = fh.get_slice(fablib)
nodes = fh.nodes_by_name(slice_)
control = nodes[cfg.CONTROL_NODE]
dynamos = nodes[cfg.DYNAMOS_NODE]
print(f"Slice {slice_.get_name()}: {slice_.get_state()}, lease end {slice_.get_lease_end()}")

In [ ]:
import shlex

assert METHOD in cfg.METHODS, f"Unknown method {METHOD!r}; choose from {list(cfg.METHODS)}"
checkout = cfg.checkout_dir(BRANCH)
installed = run(control, f"cat {cfg.INSTALLED_METHOD_FILE} 2>/dev/null || true", quiet=True).strip()
print(f"Currently installed: {installed or 'unknown/none'}")
print(f"Installing:          {METHOD} (branch {BRANCH}, checkout ~/{checkout})")
print(run(control, "helm list --all-namespaces", quiet=True))

## 3.1 Remove the current installation

In [ ]:
if RESET_FIRST:
    run(control, f"rm -f {cfg.INSTALLED_METHOD_FILE}", quiet=True)
    run_script(control, "node_scripts/reset_dynamos.sh")
    wipe = " && ".join(f"sudo rm -rf {d} && sudo mkdir -p {d} && sudo chmod 777 {d}" for d in cfg.DYNAMOS_DATA_DIRS)
    run(dynamos, wipe, quiet=True)
    print(f"Wiped {', '.join(cfg.DYNAMOS_DATA_DIRS)} on {cfg.DYNAMOS_NODE}")
else:
    print("Skipped (RESET_FIRST = False).")

## 3.2 Get the branch on `control`

In [ ]:
exists = run(control, f"test -d {checkout}/.git && echo yes || echo no", quiet=True).strip() == "yes"
if not exists:
    if not CLONE_IF_MISSING:
        raise RuntimeError(f"~/{checkout} does not exist and CLONE_IF_MISSING is False")
    run(control, f"git clone --branch {shlex.quote(BRANCH)} --single-branch {cfg.REPO_URL} {checkout}")
elif UPDATE_CHECKOUT:
    run(control, f"cd {checkout} && git fetch origin {shlex.quote(BRANCH)} && git reset --hard origin/{shlex.quote(BRANCH)}")
else:
    print(f"Reusing existing ~/{checkout} (set UPDATE_CHECKOUT = True to pull the latest commit).")

run(control, f"find {checkout} -type f -name '*.sh' -exec sed -i 's/\\r$//' {{}} + -exec chmod +x {{}} +", quiet=True)
print(run(control, f"cd {checkout} && git log -1 --format='%h %s (%cd)' && git status --short | head -20", quiet=True))

## 3.3 Install DYNAMOS for the selected method (~5–15 minutes)

In [ ]:
env = "unset DYNAMOS_ROOT; "
if IMAGE_TAG:
    env += f"export DYNAMOS_IMAGE_TAG={shlex.quote(IMAGE_TAG)}; "
monitoring_flag = "" if INSTALL_MONITORING else " --no-monitoring"

run(control, f"{env}cd $HOME/{checkout} && bash configuration/dynamos-configuration.sh fabric{monitoring_flag}")
run(control, f"echo {shlex.quote(METHOD)} > {cfg.INSTALLED_METHOD_FILE}", quiet=True)

## 3.4 Wait until DYNAMOS is ready

In [ ]:
print(run(control, "kubectl wait --for=condition=Available deployment --all --all-namespaces --timeout=15m"))
print(run(control, "kubectl rollout status statefulset/etcd -n core --timeout=10m"))
print(run(control, "kubectl get pods --all-namespaces -o wide | grep -v -E 'Running|Completed' || echo 'All pods Running/Completed'", quiet=True))

In [ ]:
print(run(control, "helm list --all-namespaces", quiet=True))
print(run(control, "kubectl get pods --all-namespaces -o wide", quiet=True))
print(run(control, "kubectl get pods --all-namespaces -o jsonpath='{range .items[*]}{.spec.containers[*].image}{\"\\n\"}{end}' | tr ' ' '\\n' | grep dynamos1 | sort -u", quiet=True))

## 3.5 Smoke test (a few rounds through the API gateway)

In [ ]:
if RUN_SMOKE_TEST:
    smoke = cfg.METHODS[METHOD]["smoke_test"].format(
        results=f"$HOME/{cfg.RESULTS_DIR}", python=f"$HOME/{cfg.EXPERIMENT_VENV}/bin/python")
    run(control, f"$HOME/{cfg.EXPERIMENT_VENV}/bin/pip install -q -r $HOME/{checkout}/{cfg.METHODS[METHOD]['requirements']}", quiet=True)
    run(control, f"mkdir -p $HOME/{cfg.RESULTS_DIR}/smoke-tests && cd $HOME/{checkout} && {smoke}")
else:
    print("Skipped (RUN_SMOKE_TEST = False).")